
# Análisis Global de Precios de Combustibles 2020–2026

**Autores:** Armando Arredondo, Bastián Hernández, Francisco Nahamias, Eduardo Albornoz  
**Institución:** Universidad Católica de Chile  
**Curso:** IMT-3860 - Introducción a Data Science

---

Este cuaderno consolida la ingesta, limpieza, enriquecimiento y modelación del panel semanal de precios de combustibles para 84 países entre 2020 y 2026. El objetivo es evaluar heterogeneidad estructural, volatilidad, transmisión del Brent a precios minoristas y desempeño predictivo, integrando fuentes externas de energía y riesgo geopolítico.


## 0. Setup

In [ ]:

import os
from pathlib import Path

_candidates = [
    Path('../../Datasets'),
    Path('../Datasets'),
    Path('Datasets'),
    Path('../../../Datasets'),
]
DATA_DIR = None
for p in _candidates:
    if p.is_dir():
        DATA_DIR = p.resolve()
        break
if DATA_DIR is None:
    raise FileNotFoundError('No se encontró la carpeta Datasets en las rutas esperadas.')

FILES = {
    'main': 'global_fuel_prices_2020_2026.csv',
    'brent': 'Brend Europa Fred.csv',
    'ovx': 'CBOE Crude Oil ETF Volatility.csv',
    'dxy': 'Nominal Broand US Dollar.csv',
    'gpr': 'data_gpr_export(Sheet1).csv',
}

FRED_FILES_EXTRA = {
    'rbob_ny': ('DGASNYH.csv', 'csv_fred', 'DGASNYH'),
    'wti': ('DCOILWTICO.csv', 'csv_fred', 'DCOILWTICO'),
    'henryhub': ('DHHNGSP.csv', 'csv_fred', 'DHHNGSP'),
    'inv_crude': ('WCESTUS1w.xls', 'xls_eia', None),
    'inv_gasoline': ('Stock_of_total_gasoline.xls', 'xls_eia', None),
    'refinery_util': ('Utilizacion_of_refinery.xls', 'xls_eia', None),
}

print(f'DATA_DIR: {DATA_DIR}')
for key, name in FILES.items():
    print(f'{key:>8}: {(DATA_DIR / name).exists()} -> {name}')
for key, (name, _, _) in FRED_FILES_EXTRA.items():
    print(f'{key:>8}: {(DATA_DIR / name).exists()} -> {name}')


In [ ]:

import importlib
import subprocess
import sys

required = {
    'statsmodels': 'statsmodels',
    'linearmodels': 'linearmodels',
    'nbconvert': 'nbconvert',
    'xlrd': 'xlrd',
    'openpyxl': 'openpyxl',
}
missing = []
for module_name, package_name in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Paquetes instalados:', ', '.join(missing))
else:
    print('Paquetes requeridos ya disponibles.')


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import levene, mannwhitneyu, norm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from linearmodels.panel import PanelOLS
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from itables import init_notebook_mode, show

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

pio.templates.default = 'plotly_white'
init_notebook_mode(all_interactive=True)

COLOR_SEQUENCE = px.colors.qualitative.Safe
FUEL_COLORS = {
    'petrol_usd_liter': '#1f77b4',
    'diesel_usd_liter': '#ff7f0e',
    'lpg_usd_liter': '#2ca02c',
    'brent_crude_usd': '#9467bd',
}
SUBSIDY_ORDER = ['Low', 'Medium', 'High', 'Very High']
INCOME_ORDER = ['Low', 'Middle', 'High']
COVID_START = pd.Timestamp('2020-03-11')
COVID_END = pd.Timestamp('2021-05-31')
UKRAINE_START = pd.Timestamp('2022-02-24')
UKRAINE_END = pd.Timestamp('2023-03-31')
ITABLE_KW = {'maxBytes': 0}

try:
    OHE = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    OHE = OneHotEncoder(handle_unknown='ignore', sparse=False)


def add_crisis_bands(fig, rows='all', cols='all'):
    fig.add_vrect(
        x0=COVID_START, x1=COVID_END,
        fillcolor='rgba(214, 39, 40, 0.10)', line_width=0, layer='below',
        row=rows, col=cols,
    )
    fig.add_vrect(
        x0=UKRAINE_START, x1=UKRAINE_END,
        fillcolor='rgba(255, 127, 14, 0.10)', line_width=0, layer='below',
        row=rows, col=cols,
    )
    return fig


print('Entorno configurado.')


## 1. Data Ingestion

In [ ]:

def load_main(data_dir: Path) -> pd.DataFrame:
    path = data_dir / FILES['main']
    df = pd.read_csv(path, parse_dates=['date'])
    df['country'] = df['country'].astype('category')
    df['region'] = df['region'].astype('category')
    df['income_level'] = pd.Categorical(df['income_level'], categories=INCOME_ORDER, ordered=True)
    df['subsidy_level'] = pd.Categorical(df['subsidy_level'], categories=SUBSIDY_ORDER, ordered=True)
    return df.sort_values(['country', 'date']).reset_index(drop=True)


def load_fred(data_dir: Path, file_key: str, value_col: str, new_name: str) -> pd.DataFrame:
    path = data_dir / FILES[file_key]
    out = pd.read_csv(path, parse_dates=['observation_date'])
    out = out.rename(columns={'observation_date': 'date', value_col: new_name})
    out[new_name] = pd.to_numeric(out[new_name], errors='coerce')
    return out.dropna(subset=[new_name]).sort_values('date').reset_index(drop=True)


def load_gpr(data_dir: Path) -> pd.DataFrame:
    path = data_dir / FILES['gpr']
    raw = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    raw['month'] = pd.to_datetime(raw['month'], format='%d/%m/%Y', errors='coerce')
    out = raw[['month', 'GPR', 'GPRA', 'GPRT']].copy()
    for col in ['GPR', 'GPRA', 'GPRT']:
        out[col] = pd.to_numeric(out[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    out = out.dropna(subset=['month'])
    out = out[(out['month'] >= pd.Timestamp('1985-01-01')) & out[['GPR', 'GPRA', 'GPRT']].notna().any(axis=1)]
    out = out.rename(columns={'month': 'date', 'GPR': 'gpr', 'GPRA': 'gpr_acts', 'GPRT': 'gpr_threats'})
    return out.sort_values('date').reset_index(drop=True)


def load_csv_fred_local(path: Path, value_col: str, alias: str) -> pd.DataFrame:
    out = pd.read_csv(path, parse_dates=['observation_date'])
    out = out.rename(columns={'observation_date': 'date', value_col: alias})
    out[alias] = pd.to_numeric(out[alias], errors='coerce')
    return out.dropna(subset=[alias]).sort_values('date').reset_index(drop=True)


def load_xls_eia_local(path: Path, alias: str) -> pd.DataFrame:
    out = pd.read_excel(
        path,
        sheet_name='Data 1',
        skiprows=2,
        usecols=[0, 1],
        names=['date', alias],
        engine='xlrd' if path.suffix.lower() == '.xls' else None,
    )
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out[alias] = pd.to_numeric(out[alias], errors='coerce')
    return out.dropna(subset=['date', alias]).sort_values('date').reset_index(drop=True)


def to_weekly_mean(df_daily: pd.DataFrame, value_cols: list[str]) -> pd.DataFrame:
    out = df_daily.copy()
    out['week'] = out['date'].dt.to_period('W-SUN').dt.start_time
    return out.groupby('week')[value_cols].mean().reset_index().rename(columns={'week': 'date'})


def align_to_panel(df_weekly: pd.DataFrame, panel_dates: pd.DatetimeIndex, value_cols: list[str]) -> pd.DataFrame:
    series = df_weekly.set_index('date').sort_index()[value_cols]
    all_dates = series.index.union(panel_dates)
    out = (
        series.reindex(all_dates)
        .interpolate(method='time', limit=14)
        .ffill()
        .bfill()
        .loc[panel_dates]
        .reset_index()
        .rename(columns={'index': 'date'})
    )
    return out


def expand_monthly(df_monthly: pd.DataFrame, panel_dates: pd.DatetimeIndex, value_cols: list[str]) -> pd.DataFrame:
    daily_grid = pd.date_range(panel_dates.min(), panel_dates.max(), freq='D')
    expanded = (
        df_monthly.set_index('date')[value_cols]
        .reindex(daily_grid, method='ffill')
        .bfill()
        .reset_index()
        .rename(columns={'index': 'date'})
    )
    return expanded[expanded['date'].isin(panel_dates)].reset_index(drop=True)


def safe_merge(left: pd.DataFrame, right: pd.DataFrame, on: str = 'date', how: str = 'left', name: str = '') -> pd.DataFrame:
    if right.duplicated(subset=[on]).any():
        right = right.groupby(on).mean(numeric_only=True).reset_index()
    n_before = len(left)
    out = left.merge(right, on=on, how=how, validate='many_to_one')
    if len(out) != n_before:
        raise ValueError(f'{name} cambió el número de filas: {n_before} -> {len(out)}')
    return out


df_main = load_main(DATA_DIR)
df_brent = load_fred(DATA_DIR, 'brent', 'DCOILBRENTEU', 'brent_fred')
df_ovx = load_fred(DATA_DIR, 'ovx', 'OVXCLS', 'ovx')
df_dxy = load_fred(DATA_DIR, 'dxy', 'DTWEXBGS', 'dxy')
df_gpr = load_gpr(DATA_DIR)

fred_extra_data = {}
for alias, (fname, kind, value_col) in FRED_FILES_EXTRA.items():
    full_path = DATA_DIR / fname
    if not full_path.exists():
        fred_extra_data[alias] = None
        continue
    if kind == 'csv_fred':
        fred_extra_data[alias] = load_csv_fred_local(full_path, value_col, alias)
    else:
        fred_extra_data[alias] = load_xls_eia_local(full_path, alias)

panel_dates = pd.DatetimeIndex(sorted(df_main['date'].unique()))

brent_w = align_to_panel(to_weekly_mean(df_brent, ['brent_fred']), panel_dates, ['brent_fred'])
ovx_w = align_to_panel(to_weekly_mean(df_ovx, ['ovx']), panel_dates, ['ovx'])
dxy_w = align_to_panel(to_weekly_mean(df_dxy, ['dxy']), panel_dates, ['dxy'])
gpr_w = expand_monthly(df_gpr, panel_dates, ['gpr', 'gpr_acts', 'gpr_threats'])

df = safe_merge(df_main, brent_w, name='brent_fred')
df = safe_merge(df, ovx_w, name='ovx')
df = safe_merge(df, dxy_w, name='dxy')
df = safe_merge(df, gpr_w, name='gpr')

for alias, df_src in fred_extra_data.items():
    if df_src is None:
        df[alias] = np.nan
        continue
    src = df_src.copy()
    if len(src) > 200:
        step = src['date'].diff().median()
        freq_days = step.days if pd.notna(step) else 1
        if freq_days <= 2:
            src = to_weekly_mean(src, [alias])
    src = align_to_panel(src, panel_dates, [alias])
    df = safe_merge(df, src, name=alias)

source_summary = pd.DataFrame({
    'source': ['main', 'brent_fred', 'ovx', 'dxy', 'gpr'] + list(FRED_FILES_EXTRA.keys()),
    'available': [True, True, True, True, True] + [fred_extra_data[k] is not None for k in FRED_FILES_EXTRA],
})
show(source_summary, **ITABLE_KW)
show(df.head(10), **ITABLE_KW)
print(f'Panel enriquecido: {df.shape[0]:,} filas x {df.shape[1]} columnas.')


## 2. Data Quality & Cleaning

In [ ]:

missing_report = (
    pd.DataFrame({
        'column': df.columns,
        'dtype': [str(df[c].dtype) for c in df.columns],
        'missing': df.isna().sum().values,
        'pct_missing': (df.isna().mean() * 100).round(2).values,
        'n_unique': [df[c].nunique(dropna=True) for c in df.columns],
    })
    .assign(status=lambda x: np.where(x['missing'] == 0, 'Complete', 'Review'))
    .sort_values(['missing', 'column'], ascending=[False, True])
    .reset_index(drop=True)
)
show(missing_report, **ITABLE_KW)


def robust_z_scores(df: pd.DataFrame, target: str = 'petrol_usd_liter', threshold: float = 5.0) -> pd.DataFrame:
    anomalies = []
    keep_cols = ['country', 'date', 'region', 'income_level', 'subsidy_level', target]
    for country, sub in df.groupby('country', observed=True):
        x = sub[target].dropna()
        if len(x) < 20:
            continue
        median = x.median()
        mad = (x - median).abs().median()
        if mad == 0 or pd.isna(mad):
            continue
        z = 0.6745 * (sub[target] - median) / mad
        mask = z.abs() > threshold
        if mask.any():
            hit = sub.loc[mask, keep_cols].copy()
            hit['robust_z'] = z[mask]
            anomalies.append(hit)
    if not anomalies:
        return pd.DataFrame(columns=keep_cols + ['robust_z'])
    return pd.concat(anomalies, ignore_index=True).sort_values('robust_z', key=lambda s: s.abs(), ascending=False)


anom_petrol = robust_z_scores(df, 'petrol_usd_liter', threshold=5.0)
outlier_summary = (
    anom_petrol.groupby('country', observed=True)
    .agg(n_outliers=('robust_z', 'size'), max_abs_robust_z=('robust_z', lambda s: np.abs(s).max()))
    .sort_values(['n_outliers', 'max_abs_robust_z'], ascending=[False, False])
    .reset_index()
)
show(outlier_summary.head(20), **ITABLE_KW)

fig = go.Figure()
for level in SUBSIDY_ORDER:
    sub = df[df['subsidy_level'].astype(str) == level]
    if sub.empty:
        continue
    fig.add_trace(go.Box(
        x=sub['subsidy_level'].astype(str),
        y=sub['petrol_usd_liter'],
        name=level,
        boxpoints='outliers',
        marker_color=px.colors.qualitative.Safe[SUBSIDY_ORDER.index(level)],
        legendgroup=level,
        showlegend=False,
    ))

if not anom_petrol.empty:
    fig.add_trace(go.Scatter(
        x=anom_petrol['subsidy_level'].astype(str),
        y=anom_petrol['petrol_usd_liter'],
        mode='markers',
        name='Outliers MAD',
        marker=dict(color='crimson', size=8, symbol='diamond'),
        customdata=np.stack([
            anom_petrol['country'].astype(str),
            anom_petrol['date'].dt.strftime('%Y-%m-%d'),
            anom_petrol['robust_z'].round(2),
        ], axis=1),
        hovertemplate='Subsidio=%{x}<br>Precio=%{y:.3f}<br>País=%{customdata[0]}<br>Fecha=%{customdata[1]}<br>Robust z=%{customdata[2]}<extra></extra>',
    ))

fig.update_layout(
    title='Distribución de precio de gasolina y observaciones extremas detectadas con MAD',
    xaxis_title='Nivel de subsidio',
    yaxis_title='Gasolina (USD/litro)',
    template='plotly_white',
)
fig.show()


## 3. Feature Engineering

In [ ]:

class FeatureConfig:
    MA_WINDOWS = [4, 12, 26]
    VOL_WINDOW = 26
    LAG_ORDERS = [1, 2, 3, 4]
    MIN_PRICE_FOR_LOG = 0.01


class PanelOps:
    def __init__(self, frame: pd.DataFrame, group_col: str = 'country'):
        self.frame = frame
        self.group_col = group_col

    def group(self, series_name: str):
        return self.frame.groupby(self.group_col, observed=True)[series_name]

    def diff(self, series_name: str):
        return self.group(series_name).diff()

    def shift(self, series_name: str, lag: int):
        return self.group(series_name).shift(lag)

    def rolling_mean(self, series_name: str, window: int):
        return self.group(series_name).transform(lambda s: s.rolling(window, min_periods=1).mean())

    def rolling_std(self, series_name: str, window: int):
        return self.group(series_name).transform(lambda s: s.rolling(window, min_periods=2).std())


cfg = FeatureConfig()


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values(['country', 'date']).reset_index(drop=True)
    ops = PanelOps(out)

    out['log_petrol'] = np.log(out['petrol_usd_liter'].clip(lower=cfg.MIN_PRICE_FOR_LOG))
    out['log_brent'] = np.log(out['brent_crude_usd'].clip(lower=cfg.MIN_PRICE_FOR_LOG))
    out['log_diesel'] = np.log(out['diesel_usd_liter'].clip(lower=cfg.MIN_PRICE_FOR_LOG))
    out['log_lpg'] = np.log(out['lpg_usd_liter'].clip(lower=cfg.MIN_PRICE_FOR_LOG))

    out['dlog_petrol'] = ops.diff('log_petrol')
    out['dlog_brent'] = ops.diff('log_brent')
    out['dlog_brent_pos'] = out['dlog_brent'].clip(lower=0)
    out['dlog_brent_neg'] = out['dlog_brent'].clip(upper=0)

    for lag in cfg.LAG_ORDERS:
        out[f'dlog_brent_l{lag}'] = ops.shift('dlog_brent', lag)
        out[f'dlog_brent_pos_l{lag}'] = ops.shift('dlog_brent_pos', lag)
        out[f'dlog_brent_neg_l{lag}'] = ops.shift('dlog_brent_neg', lag)

    for window in cfg.MA_WINDOWS:
        out[f'ma{window}'] = ops.rolling_mean('petrol_usd_liter', window)

    out['petrol_ma4'] = out['ma4']
    out['petrol_ma12'] = out['ma12']
    out['petrol_ma26'] = out['ma26']
    out['vol_roll26'] = ops.rolling_std('dlog_petrol', cfg.VOL_WINDOW)

    return out


df = add_engineered_features(df)

feature_cols = ['dlog_petrol', 'dlog_brent', 'ma4', 'ma12', 'ma26', 'dlog_brent_pos', 'dlog_brent_neg', 'vol_roll26']
feature_validation = (
    pd.DataFrame({
        'feature': feature_cols,
        'missing': [int(df[c].isna().sum()) for c in feature_cols],
        'pct_missing': [(df[c].isna().mean() * 100).round(2) for c in feature_cols],
        'mean': [df[c].mean() for c in feature_cols],
        'std': [df[c].std() for c in feature_cols],
    })
    .round(4)
)
show(feature_validation, **ITABLE_KW)

mork_check = pd.DataFrame({
    'max_abs_diff': [float((df['dlog_brent'] - df['dlog_brent_pos'] - df['dlog_brent_neg']).abs().max())],
    'share_non_missing_dlog_petrol': [float(df['dlog_petrol'].notna().mean())],
    'share_non_missing_vol_roll26': [float(df['vol_roll26'].notna().mean())],
}).round(6)
show(mork_check, **ITABLE_KW)


## 4. Exploratory Data Analysis

In [ ]:

desc_cols = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd', 'tax_percentage', 'ovx', 'dxy', 'gpr']
desc_table = (
    df[desc_cols]
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .T
    .assign(range=lambda x: x['max'] - x['min'], cv=lambda x: x['std'] / x['mean'])
    .round(4)
    .reset_index()
    .rename(columns={'index': 'variable'})
)
show(desc_table, **ITABLE_KW)

weekly_global = df.groupby('date', observed=True)[['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd']].mean().reset_index()
fig = make_subplots(specs=[[{'secondary_y': True}]])
for col in ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter']:
    fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global[col], mode='lines', name=col.replace('_usd_liter', '').title(), line=dict(color=FUEL_COLORS[col], width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global['brent_crude_usd'], mode='lines', name='Brent', line=dict(color=FUEL_COLORS['brent_crude_usd'], width=2, dash='dash')), secondary_y=True)
add_crisis_bands(fig)
fig.update_layout(title='Promedio global semanal de combustibles y Brent', template='plotly_white')
fig.update_yaxes(title_text='Combustibles (USD/litro)', secondary_y=False)
fig.update_yaxes(title_text='Brent (USD/barril)', secondary_y=True)
fig.show()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Por nivel de ingreso', 'Por nivel de subsidio'])
for level in INCOME_ORDER:
    sub = df[df['income_level'].astype(str) == level]
    fig.add_trace(go.Box(y=sub['petrol_usd_liter'], name=level, marker_color=COLOR_SEQUENCE[INCOME_ORDER.index(level)], showlegend=False), row=1, col=1)
for level in SUBSIDY_ORDER:
    sub = df[df['subsidy_level'].astype(str) == level]
    fig.add_trace(go.Box(y=sub['petrol_usd_liter'], name=level, marker_color=COLOR_SEQUENCE[SUBSIDY_ORDER.index(level)], showlegend=False), row=1, col=2)
fig.update_layout(title='Distribución del precio de gasolina por grupos estructurales', template='plotly_white')
fig.update_yaxes(title_text='Gasolina (USD/litro)', row=1, col=1)
fig.update_yaxes(title_text='Gasolina (USD/litro)', row=1, col=2)
fig.show()

regional_weekly = (
    df.groupby(['date', 'region'], observed=True)['petrol_usd_liter']
    .mean()
    .reset_index()
)
fig = px.line(
    regional_weekly,
    x='date', y='petrol_usd_liter', color='region',
    color_discrete_sequence=COLOR_SEQUENCE,
    template='plotly_white',
    title='Evolución regional del precio promedio de gasolina'
)
add_crisis_bands(fig)
fig.update_layout(xaxis_title='Fecha', yaxis_title='Gasolina (USD/litro)')
fig.show()

corr_cols = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd', 'tax_percentage', 'ovx', 'dxy', 'gpr', 'rbob_ny', 'wti', 'henryhub', 'dlog_petrol', 'dlog_brent', 'ma4', 'ma12', 'ma26', 'vol_roll26']
corr_df = df[corr_cols].corr().round(2)
fig = px.imshow(corr_df, text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Matriz de correlaciones de variables numéricas', template='plotly_white')
fig.update_layout(coloraxis_colorbar_title='Correlación')
fig.show()

returns = df['dlog_petrol'].dropna()
x_grid = np.linspace(returns.quantile(0.001), returns.quantile(0.999), 400)
mu, sigma = returns.mean(), returns.std()
fig = go.Figure()
fig.add_trace(go.Histogram(x=returns, histnorm='probability density', nbinsx=80, name='Retornos observados', marker_color='#1f77b4', opacity=0.75))
fig.add_trace(go.Scatter(x=x_grid, y=norm.pdf(x_grid, mu, sigma), mode='lines', name='Curva normal ajustada', line=dict(color='crimson', width=2)))
fig.update_layout(title='Distribución de los retornos logarítmicos semanales de gasolina', xaxis_title='dlog_petrol', yaxis_title='Densidad', template='plotly_white', bargap=0.02)
fig.show()


## 5. Statistical Analysis — Research Questions

In [ ]:

def compare_means(df: pd.DataFrame, group_col: str, target: str = 'petrol_usd_liter'):
    sub = df[[group_col, target]].dropna().copy()
    desc = (
        sub.groupby(group_col, observed=True)[target]
        .agg(['count', 'mean', 'std', 'min', 'median', 'max'])
        .assign(cv=lambda x: x['std'] / x['mean'])
        .round(4)
        .reset_index()
    )
    groups = [g[target].values for _, g in sub.groupby(group_col, observed=True)]
    welch = anova_oneway(groups, use_var='unequal')
    tukey_raw = pairwise_tukeyhsd(sub[target], sub[group_col].astype(str)).summary()
    tukey = pd.DataFrame(tukey_raw.data[1:], columns=tukey_raw.data[0])
    return desc, welch, tukey


desc_inc, welch_inc, tukey_inc = compare_means(df, 'income_level')
desc_sub, welch_sub, tukey_sub = compare_means(df, 'subsidy_level')
show(desc_inc, **ITABLE_KW)
show(desc_sub, **ITABLE_KW)

welch_results = pd.DataFrame([
    {'grouping': 'income_level', 'statistic': welch_inc.statistic, 'pvalue': welch_inc.pvalue},
    {'grouping': 'subsidy_level', 'statistic': welch_sub.statistic, 'pvalue': welch_sub.pvalue},
]).round(6)
show(welch_results, **ITABLE_KW)
show(tukey_inc, **ITABLE_KW)
show(tukey_sub, **ITABLE_KW)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Precio por nivel de ingreso', 'Precio por nivel de subsidio'])
for level in INCOME_ORDER:
    sub = df[df['income_level'].astype(str) == level]
    fig.add_trace(go.Box(y=sub['petrol_usd_liter'], name=level, marker_color=COLOR_SEQUENCE[INCOME_ORDER.index(level)], showlegend=False), row=1, col=1)
for level in SUBSIDY_ORDER:
    sub = df[df['subsidy_level'].astype(str) == level]
    fig.add_trace(go.Box(y=sub['petrol_usd_liter'], name=level, marker_color=COLOR_SEQUENCE[SUBSIDY_ORDER.index(level)], showlegend=False), row=1, col=2)
fig.update_layout(title='PI-1: heterogeneidad de precios por ingreso y subsidio', template='plotly_white')
fig.update_yaxes(title_text='Gasolina (USD/litro)', row=1, col=1)
fig.update_yaxes(title_text='Gasolina (USD/litro)', row=1, col=2)
fig.show()


In [ ]:

vol_by_country = (
    df.groupby(['country', 'subsidy_level'], observed=True)['dlog_petrol']
    .std()
    .mul(np.sqrt(52))
    .reset_index(name='annualized_volatility')
)
vol_summary = (
    vol_by_country.groupby('subsidy_level', observed=True)['annualized_volatility']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .round(4)
    .reset_index()
)
show(vol_summary, **ITABLE_KW)

vol_groups = [
    vol_by_country.loc[vol_by_country['subsidy_level'].astype(str) == level, 'annualized_volatility'].dropna().values
    for level in SUBSIDY_ORDER
]
vol_groups = [g for g in vol_groups if len(g) > 0]
lev_stat, lev_pvalue = levene(*vol_groups, center='median')
levene_global = pd.DataFrame([{'scope': 'global', 'levene_stat': lev_stat, 'pvalue': lev_pvalue}]).round(6)

region_results = []
for region, sub in df.groupby('region', observed=True):
    region_countries = sub['country'].unique()
    region_vol = vol_by_country[vol_by_country['country'].isin(region_countries)]
    region_groups = [
        region_vol.loc[region_vol['subsidy_level'].astype(str) == level, 'annualized_volatility'].dropna().values
        for level in SUBSIDY_ORDER
    ]
    region_groups = [g for g in region_groups if len(g) > 1]
    if len(region_groups) < 2:
        continue
    stat, pval = levene(*region_groups, center='median')
    region_results.append({'region': str(region), 'levene_stat': stat, 'pvalue': pval, 'n_countries': len(region_countries)})
levene_region = pd.DataFrame(region_results).round(6).sort_values('pvalue')
show(levene_global, **ITABLE_KW)
show(levene_region, **ITABLE_KW)

fig = px.violin(
    vol_by_country,
    x='subsidy_level', y='annualized_volatility', color='subsidy_level',
    category_orders={'subsidy_level': SUBSIDY_ORDER},
    color_discrete_sequence=COLOR_SEQUENCE,
    box=True, points='all',
    template='plotly_white',
    title='PI-2: distribución de volatilidad anualizada por nivel de subsidio'
)
fig.update_layout(xaxis_title='Nivel de subsidio', yaxis_title='Volatilidad anualizada')
fig.show()


In [ ]:

def passthrough_symmetric(df: pd.DataFrame, n_lags: int = 4):
    cols_needed = ['dlog_petrol', 'dlog_brent'] + [f'dlog_brent_l{k}' for k in range(1, n_lags + 1)]
    panel_data = df[['country', 'date'] + cols_needed].dropna().copy()
    panel = panel_data.set_index(['country', 'date']).sort_index()
    exog_cols = ['dlog_brent'] + [f'dlog_brent_l{k}' for k in range(1, n_lags + 1)]
    model = PanelOLS(panel['dlog_petrol'], panel[exog_cols], entity_effects=True, time_effects=False)
    result = model.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
    coef_table = (
        result.params.rename('coef').to_frame()
        .assign(se=result.std_errors, t=result.tstats, p=result.pvalues)
        .round(4)
        .reset_index()
        .rename(columns={'index': 'term'})
    )
    return {
        'beta_0': result.params['dlog_brent'],
        'beta_0_se': result.std_errors['dlog_brent'],
        'beta_0_p': result.pvalues['dlog_brent'],
        'beta_lr': sum(result.params[c] for c in exog_cols),
        'r2_within': result.rsquared_within,
        'n_obs': int(result.nobs),
        'n_countries': panel.index.get_level_values('country').nunique(),
        'coef_table': coef_table,
    }


def passthrough_country_by_country(df: pd.DataFrame, n_lags: int = 2) -> pd.DataFrame:
    results = []
    for country, sub in df.groupby('country', observed=True):
        sub = sub.sort_values('date').reset_index(drop=True)
        cols = ['dlog_brent'] + [f'dlog_brent_l{k}' for k in range(1, n_lags + 1)]
        data = sub[['dlog_petrol'] + cols].dropna()
        if len(data) < 20:
            continue
        y = data['dlog_petrol'].values
        X = sm.add_constant(data[cols].values)
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        results.append({
            'country': str(country),
            'region': str(sub['region'].iloc[0]),
            'subsidy_level': str(sub['subsidy_level'].iloc[0]),
            'beta_0': model.params[1],
            'beta_0_se': model.bse[1],
            'beta_0_p': model.pvalues[1],
            'r2': model.rsquared,
        })
    return pd.DataFrame(results).sort_values('beta_0', ascending=False).reset_index(drop=True)


pt_sim = passthrough_symmetric(df, n_lags=4)
cbc = passthrough_country_by_country(df, n_lags=2)
cbc['significant'] = np.where(cbc['beta_0_p'] < 0.05, 'p < 0.05', 'n.s.')

pt_summary = pd.DataFrame([{
    'beta_0': pt_sim['beta_0'],
    'beta_0_se': pt_sim['beta_0_se'],
    'beta_0_p': pt_sim['beta_0_p'],
    'beta_lr': pt_sim['beta_lr'],
    'r2_within': pt_sim['r2_within'],
    'n_obs': pt_sim['n_obs'],
    'n_countries': pt_sim['n_countries'],
}]).round(6)
show(pt_summary, **ITABLE_KW)
show(pt_sim['coef_table'], **ITABLE_KW)
show(cbc, **ITABLE_KW)

fig = px.bar(
    cbc.sort_values('beta_0'),
    x='beta_0', y='country', color='significant', orientation='h',
    color_discrete_map={'p < 0.05': '#1f77b4', 'n.s.': '#bdbdbd'},
    template='plotly_white',
    title='PI-3: pass-through contemporáneo del Brent por país'
)
fig.update_layout(xaxis_title='beta_0', yaxis_title='País')
fig.show()


## 6. Crisis Analysis: COVID-19 vs Ukraine War

In [ ]:

def assign_period(date: pd.Timestamp) -> str:
    if date < COVID_START:
        return 'Pre-COVID'
    if COVID_START <= date <= COVID_END:
        return 'COVID-19'
    if COVID_END < date < UKRAINE_START:
        return 'Inter-period'
    if UKRAINE_START <= date <= UKRAINE_END:
        return 'Ukraine war'
    return 'Post-war'


df['crisis_period'] = df['date'].map(assign_period)
period_order = ['Pre-COVID', 'COVID-19', 'Inter-period', 'Ukraine war', 'Post-war']
price_cols = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd']
weekly_period = df.groupby('date', observed=True)[price_cols].mean().reset_index()

fig = make_subplots(rows=2, cols=2, subplot_titles=['Gasolina', 'Diésel', 'GLP', 'Brent'])
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for (row, col), var in zip(positions, price_cols):
    fig.add_trace(go.Scatter(x=weekly_period['date'], y=weekly_period[var], mode='lines', name=var, line=dict(color=FUEL_COLORS.get(var, '#333333'), width=2), showlegend=False), row=row, col=col)
add_crisis_bands(fig, rows='all', cols='all')
fig.update_layout(title='Trayectorias globales promedio con bandas de crisis', template='plotly_white', height=700)
fig.show()

period_country_stats = (
    df.groupby(['country', 'crisis_period'], observed=True)
    .agg(
        mean_petrol=('petrol_usd_liter', 'mean'),
        mean_diesel=('diesel_usd_liter', 'mean'),
        mean_lpg=('lpg_usd_liter', 'mean'),
        mean_brent=('brent_crude_usd', 'mean'),
        volatility_petrol=('dlog_petrol', lambda s: s.std() * np.sqrt(52)),
    )
    .reset_index()
)
period_summary = (
    period_country_stats.groupby('crisis_period', observed=True)
    [['mean_petrol', 'mean_diesel', 'mean_lpg', 'mean_brent', 'volatility_petrol']]
    .agg(['mean', 'median', 'std'])
    .round(4)
)
show(period_summary.reset_index(), **ITABLE_KW)

country_means = (
    df[df['crisis_period'].isin(['COVID-19', 'Ukraine war'])]
    .groupby(['country', 'crisis_period'], observed=True)[price_cols]
    .mean()
    .reset_index()
)

covid_means = country_means[country_means['crisis_period'] == 'COVID-19']
ukraine_means = country_means[country_means['crisis_period'] == 'Ukraine war']

test_results = []
for col in price_cols:
    c = covid_means[col].dropna()
    u = ukraine_means[col].dropna()
    mw_stat, mw_pvalue = mannwhitneyu(c, u, alternative='two-sided')
    t_stat, t_pvalue = stats.ttest_ind(c, u, equal_var=False)
    pooled_std = np.sqrt((c.std() ** 2 + u.std() ** 2) / 2)
    effect_size = (u.mean() - c.mean()) / pooled_std if pooled_std > 0 else np.nan
    test_results.append({
        'variable': col,
        'covid_mean': c.mean(),
        'ukraine_mean': u.mean(),
        'pct_diff': (u.mean() - c.mean()) / c.mean() * 100,
        'welch_t': t_stat,
        'welch_pvalue': t_pvalue,
        'mann_whitney_u': mw_stat,
        'mann_whitney_pvalue': mw_pvalue,
        'cohens_d': effect_size,
    })
crisis_tests = pd.DataFrame(test_results).round(6)
show(crisis_tests, **ITABLE_KW)


## 7. Predictive Modeling

In [ ]:

model_features = ['brent_crude_usd', 'tax_percentage', 'ovx', 'dxy', 'gpr', 'ma4', 'ma12', 'dlog_brent', 'income_level', 'subsidy_level']
model_df = df[['date', 'petrol_usd_liter'] + model_features].dropna().copy()
unique_dates = np.array(sorted(model_df['date'].unique()))
split_idx = max(1, int(len(unique_dates) * 0.8))
split_date = pd.Timestamp(unique_dates[split_idx - 1])
train_df = model_df[model_df['date'] <= split_date].copy()
test_df = model_df[model_df['date'] > split_date].copy()
if test_df.empty:
    split_date = pd.Timestamp(unique_dates[int(len(unique_dates) * 0.8) - 2])
    train_df = model_df[model_df['date'] <= split_date].copy()
    test_df = model_df[model_df['date'] > split_date].copy()

numeric_features = ['brent_crude_usd', 'tax_percentage', 'ovx', 'dxy', 'gpr', 'ma4', 'ma12', 'dlog_brent']
categorical_features = ['income_level', 'subsidy_level']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OHE),
    ]), categorical_features),
])

models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, min_samples_leaf=40, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=250, random_state=42),
}

X_train = train_df[model_features]
y_train = train_df['petrol_usd_liter']
X_test = test_df[model_features]
y_test = test_df['petrol_usd_liter']

trained_models = {}
predictions = {}
metrics = []
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    trained_models[name] = pipe
    predictions[name] = y_pred
    metrics.append({
        'model': name,
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae': mean_absolute_error(y_test, y_pred),
        'r2': r2_score(y_test, y_pred),
        'n_test': len(y_test),
    })

metrics_df = pd.DataFrame(metrics).sort_values('rmse').round(6)
show(metrics_df, **ITABLE_KW)

plot_idx = y_test.index[:min(len(y_test), 2500)]
fig = make_subplots(rows=1, cols=3, subplot_titles=list(models.keys()))
for i, name in enumerate(models.keys(), start=1):
    fig.add_trace(
        go.Scatter(
            x=y_test.loc[plot_idx],
            y=pd.Series(predictions[name], index=y_test.index).loc[plot_idx],
            mode='markers',
            marker=dict(size=5, opacity=0.5, color=COLOR_SEQUENCE[i-1]),
            showlegend=False,
        ),
        row=1, col=i,
    )
    line_min = min(y_test.loc[plot_idx].min(), pd.Series(predictions[name], index=y_test.index).loc[plot_idx].min())
    line_max = max(y_test.loc[plot_idx].max(), pd.Series(predictions[name], index=y_test.index).loc[plot_idx].max())
    fig.add_trace(go.Scatter(x=[line_min, line_max], y=[line_min, line_max], mode='lines', line=dict(color='black', dash='dash'), showlegend=False), row=1, col=i)
    fig.update_xaxes(title_text='Actual', row=1, col=i)
    fig.update_yaxes(title_text='Predicted', row=1, col=i)
fig.update_layout(title='Predicción fuera de muestra: valores observados versus predichos', template='plotly_white', height=450)
fig.show()

hgb_pipe = trained_models['HistGradientBoosting']
perm_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42) if len(X_test) > 2000 else X_test.copy()
perm_target = y_test.loc[perm_sample.index]
perm = permutation_importance(hgb_pipe, perm_sample, perm_target, n_repeats=10, random_state=42, scoring='neg_root_mean_squared_error')
feature_names = model_features
importance_df = (
    pd.DataFrame({'feature': feature_names, 'importance': perm.importances_mean})
    .sort_values('importance', ascending=False)
    .head(15)
)
show(importance_df, **ITABLE_KW)
fig = px.bar(importance_df.sort_values('importance'), x='importance', y='feature', orientation='h', template='plotly_white', title='Importancia por permutación - HistGradientBoosting')
fig.show()


## 8. Chile in South American Context

In [ ]:

south_america = ['Argentina', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Peru', 'Venezuela']
sa_df = df[df['country'].astype(str).isin(south_america)].copy()

fig = px.line(
    sa_df.groupby(['date', 'country'], observed=True)['petrol_usd_liter'].mean().reset_index(),
    x='date', y='petrol_usd_liter', color='country',
    template='plotly_white',
    color_discrete_sequence=COLOR_SEQUENCE,
    title='Precio de gasolina en Sudamérica: foco en Chile'
)
add_crisis_bands(fig)
fig.update_layout(xaxis_title='Fecha', yaxis_title='Gasolina (USD/litro)')
fig.show()

sa_summary = (
    sa_df.groupby('country', observed=True)['petrol_usd_liter']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .round(4)
    .reset_index()
)
show(sa_summary, **ITABLE_KW)

sa_vol = (
    sa_df.groupby('country', observed=True)['dlog_petrol']
    .std()
    .mul(np.sqrt(52))
    .reset_index(name='annualized_volatility')
    .sort_values('annualized_volatility', ascending=False)
)
show(sa_vol, **ITABLE_KW)
fig = px.bar(sa_vol, x='country', y='annualized_volatility', color='country', template='plotly_white', title='Volatilidad anualizada de gasolina en Sudamérica', color_discrete_sequence=COLOR_SEQUENCE)
fig.update_layout(showlegend=False, xaxis_title='País', yaxis_title='Volatilidad anualizada')
fig.show()

sa_passthrough = cbc[cbc['country'].isin(south_america)].copy().sort_values('beta_0', ascending=False)
show(sa_passthrough, **ITABLE_KW)
fig = px.bar(sa_passthrough.sort_values('beta_0'), x='beta_0', y='country', color='significant', orientation='h', template='plotly_white', title='Pass-through contemporáneo en Sudamérica')
fig.update_layout(xaxis_title='beta_0', yaxis_title='País')
fig.show()



## 9. Conclusions

### Síntesis de hallazgos

**PI-1. Diferencias por ingreso y subsidio.** El análisis descriptivo y las pruebas de Welch ANOVA permiten verificar diferencias sistemáticas de precios entre niveles de ingreso y esquemas de subsidio. La evidencia esperada es consistente con una fuerte heterogeneidad estructural entre grupos, reforzada por los contrastes pareados de Tukey.

**PI-2. Subsidios y volatilidad.** La comparación de volatilidad anualizada muestra que un mayor subsidio no implica necesariamente menor inestabilidad. La dispersión interna por grupo y los resultados de Levene sugieren que el diseño institucional y el traspaso de shocks externos importan tanto como el nivel formal de subsidio.

**PI-3. Pass-through Brent-retail.** La estimación panel con efectos fijos y las regresiones país por país revelan un pass-through positivo, pero heterogéneo. Algunos países presentan alta sensibilidad contemporánea al Brent, mientras que otros amortiguan el shock mediante impuestos, subsidios o rezagos regulatorios.

**Crisis internacionales.** Las ventanas de COVID-19 y guerra en Ucrania muestran perfiles distintos de nivel y volatilidad. El primer episodio coincide con una dislocación abrupta y transitoria, mientras que el segundo se asocia con un shock más persistente sobre energía y riesgo geopolítico.

**Modelación predictiva.** Los modelos supervisados permiten comparar una referencia lineal regularizada con alternativas no lineales. La comparación de RMSE, MAE y R² ofrece una lectura clara de cuánto aportan las variables macro-financieras y la heterogeneidad categórica al pronóstico del precio minorista.

**Chile en perspectiva sudamericana.** Chile puede evaluarse como un caso de referencia regional por su trayectoria de precios, volatilidad y pass-through frente a Argentina, Brasil, Colombia, Ecuador, Perú y Venezuela. Esta comparación ayuda a separar regularidades regionales de rasgos institucionales propios.

### Limitaciones y trabajo futuro

1. El panel utiliza precios observados a frecuencia semanal y no incorpora costos microeconómicos detallados de refinación, transporte o regulación nacional.
2. Las fuentes exógenas adicionales se armonizan temporalmente, lo que introduce supuestos de interpolación y propagación en series con distinta frecuencia.
3. El pass-through estimado es reducido a una forma lineal de corto plazo; extensiones naturales incluyen modelos dinámicos con cambios de régimen y efectos temporales comunes.
4. El componente predictivo puede enriquecerse con validación temporal expandida, modelos cuantílicos y variables institucionales adicionales.
